In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: A Qiskit Function by Q-CTRL Fire Opal

*Lihat [rujukan API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Fungsi Qiskit adalah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui API Platform IBM Quantum). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.


<Accordion>
<AccordionItem title="Versi pakej">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Gambaran keseluruhan
Dengan Fire Opal Optimization Solver, anda boleh menyelesaikan masalah pengoptimuman skala utiliti pada perkakasan kuantum tanpa memerlukan kepakaran kuantum. Cukup masukkan definisi masalah peringkat tinggi, dan Solver akan menguruskan selebihnya. Keseluruhan aliran kerja ini peka hingar dan memanfaatkan [Pengurusan Prestasi Fire Opal](/guides/q-ctrl-performance-management) di sebaliknya. Solver secara konsisten memberikan penyelesaian tepat untuk masalah yang mencabar secara klasik, walaupun pada skala penuh peranti pada QPU IBM&reg; yang terbesar.

Solver fleksibel dan boleh digunakan untuk menyelesaikan masalah pengoptimuman kombinatorial yang ditakrifkan sebagai fungsi objektif atau graf arbitrari. Masalah tidak perlu dipetakan ke topologi peranti. Masalah tanpa kekangan dan dengan kekangan boleh diselesaikan, dengan syarat kekangan boleh diformulasikan sebagai terma penalti. Contoh-contoh dalam panduan ini menunjukkan cara menyelesaikan masalah pengoptimuman skala utiliti tanpa kekangan dan dengan kekangan menggunakan jenis input Solver yang berbeza. Contoh pertama melibatkan masalah max-cut yang ditakrifkan pada graf 3-Biasa 156 nod, manakala contoh kedua menangani masalah Penutup Bucu Minimum 50 nod yang ditakrifkan oleh fungsi kos.

Untuk mendapatkan akses kepada Optimization Solver, [hubungi Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Penerangan fungsi
Solver mengoptimumkan dan mengautomatikkan keseluruhan algoritma sepenuhnya, dari penindasan ralat pada peringkat perkakasan hingga pemetaan masalah yang cekap dan pengoptimuman klasik gelung tertutup. Di sebalik tabir, saluran Solver mengurangkan ralat pada setiap peringkat, membolehkan peningkatan prestasi yang diperlukan untuk menskalakan secara bermakna. Aliran kerja asas ini terinspirasi oleh Quantum Approximate Optimization Algorithm (QAOA), yang merupakan algoritma hibrid kuantum-klasik. Untuk ringkasan terperinci aliran kerja Optimization Solver yang penuh, rujuk [manuskrip yang diterbitkan](https://arxiv.org/abs/2406.01743).

![Visualisasi aliran kerja Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Untuk menyelesaikan masalah umum dengan Optimization Solver:
1. Takrifkan masalah anda sebagai fungsi objektif, graf, atau rantai spin `SparsePauliOp`.
2. Sambung ke fungsi melalui Katalog Fungsi Qiskit.
3. Jalankan masalah dengan Solver dan dapatkan semula keputusan.
### Format masalah yang diterima
- Representasi ungkapan polinomial bagi fungsi objektif. Sebaik-baiknya dibuat dalam Python dengan objek SymPy Poly sedia ada dan diformat menjadi rentetan menggunakan [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Representasi graf bagi jenis masalah tertentu. Graf perlu dibuat menggunakan pustaka networkx dalam Python. Ia kemudian ditukar kepada rentetan menggunakan fungsi networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Representasi rantai spin bagi masalah tertentu. Rantai spin perlu direpresentasikan sebagai objek `SparsePauliOp`; lihat [dokumentasi](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) untuk maklumat lanjut.

> **Note:** Jika anda ingin menggunakan backend yang tidak disokong oleh fungsi ini pada masa ini, [hubungi Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) untuk menambah sokongan.
## Penanda aras
[Keputusan penanda aras yang diterbitkan](https://arxiv.org/abs/2406.01743) menunjukkan bahawa Solver berjaya menyelesaikan masalah dengan lebih 120 qubit, malah mengatasi keputusan yang diterbitkan sebelumnya pada peranti anil kuantum dan ion perangkap. Metrik penanda aras berikut memberikan petunjuk kasar tentang ketepatan dan penskalaan jenis masalah berdasarkan beberapa contoh. Metrik sebenar mungkin berbeza berdasarkan pelbagai ciri masalah, seperti bilangan terma dalam fungsi objektif (ketumpatan) dan lokaliti, bilangan pemboleh ubah, dan tertib polinomial.

"Bilangan qubit" yang ditunjukkan bukan had keras tetapi mewakili ambang kasar di mana anda boleh mengharapkan ketepatan penyelesaian yang sangat konsisten. Saiz masalah yang lebih besar telah berjaya diselesaikan, dan ujian melebihi had ini digalakkan.

Kesambungan qubit arbitrari disokong merentasi semua jenis masalah.

| Jenis masalah    | Bilangan qubit | Contoh | Ketepatan | Jumlah masa (s) | Penggunaan runtime (s) | Bilangan iterasi
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Masalah kuadratik bersambung jarang  | 156 | max-cut 3-regular| 100%     | 1764     | 293          | 16 |
| Pengoptimuman binari tertib tinggi | 156 | Model kaca spin Ising | 100%      | 1461     | 272          | 16 |
| Masalah kuadratik bersambung padat | 50 | max-cut bersambung penuh| 100%      |  1758    | 268  | 12 |
| Masalah dengan kekangan dan terma penalti | 50 | Penutup Bucu Minimum Berwajaran dengan ketumpatan tepi 8% | 100%      | 1074     | 215 | 10 |
## Mulakan
Pertama, sahkan menggunakan [kunci API IBM Quantum](http://quantum.cloud.ibm.com/) anda. Kemudian, pilih Fungsi Qiskit seperti berikut. (Petikan kod ini menganggap anda sudah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Takrifkan masalah
Anda boleh menjalankan masalah Max-Cut dengan mentakrifkan masalah graf dan menentukan `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Jalankan masalah
Apabila menggunakan kaedah input berasaskan graf, tentukan jenis masalah.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. Dapatkan semula keputusan
Dapatkan semula nilai potongan optimum dari kamus keputusan.

> **Note:** Pemetaan pemboleh ubah ke bitstring mungkin telah berubah. Kamus output mengandungi subdiksi `variables_to_bitstring_index_map`, yang membantu mengesahkan urutan.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

Anda boleh mengesahkan ketepatan keputusan dengan menyelesaikan masalah secara klasik menggunakan penyelesai sumber terbuka seperti [PuLP](https://coin-or.github.io/pulp/) jika graf tidak bersambung padat. Masalah berketumpatan tinggi mungkin memerlukan penyelesai klasik lanjutan untuk mengesahkan penyelesaian.
## Contoh: Pengoptimuman dengan kekangan
Contoh max-cut sebelumnya adalah masalah pengoptimuman binari kuadratik tanpa kekangan yang biasa. Optimization Solver Q-CTRL boleh digunakan untuk pelbagai jenis masalah, termasuk pengoptimuman dengan kekangan. Anda boleh menyelesaikan jenis masalah arbitrari dengan memasukkan definisi masalah yang direpresentasikan sebagai polinomial di mana kekangan dimodelkan sebagai terma penalti.

Contoh berikut menunjukkan cara membina fungsi kos untuk masalah pengoptimuman dengan kekangan, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Selain pakej `qiskit-ibm-catalog` dan `qiskit`, anda juga akan menggunakan pakej berikut untuk menjalankan contoh ini: `numpy`, `networkx`, dan `sympy`. Anda boleh memasang pakej ini dengan menyahkomenter sel berikut jika anda menjalankan contoh ini dalam notebook menggunakan kernel IPython.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Takrifkan masalah
Takrifkan masalah MVC rawak dengan menjana graf dengan nod berwajaran rawak.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![Output of the previous code cell](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Model pengoptimuman standard untuk MVC berwajaran boleh diformulasikan seperti berikut. Pertama, penalti mesti ditambah untuk sebarang kes di mana tepi tidak disambungkan kepada bucu dalam subset. Oleh itu, biarkan $n_i = 1$ jika bucu $i$ berada dalam penutup (iaitu, dalam subset) dan $n_i = 0$ sebaliknya. Kedua, matlamat adalah untuk meminimumkan jumlah bilangan bucu dalam subset, yang boleh direpresentasikan oleh fungsi berikut:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Kini setiap tepi dalam graf mestilah mengandungi sekurang-kurangnya satu titik hujung dari penutup, yang boleh dinyatakan sebagai ketaksamaan:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Sebarang kes di mana tepi tidak disambungkan kepada bucu penutup mesti dikenakan penalti. Ini boleh direpresentasikan dalam fungsi kos dengan menambah penalti berbentuk $P(1-n_i-n_j+n_i n_j)$ di mana $P$ adalah pemalar penalti positif. Oleh itu, alternatif tanpa kekangan kepada ketaksamaan berlandaskan kekangan untuk MVC berwajaran ialah:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Jalankan masalah

In [17]:
print(mvc_job.status())

QUEUED


Semak [status](/guides/functions#check-job-status) beban kerja Fungsi Qiskit anda atau dapatkan semula [keputusan](/guides/functions#retrieve-results) seperti berikut:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>